# 7.2 多模态端侧部署

视觉语言模型（VLM）、语音模型等多模态模型的端侧部署，需要对不同模态的编码器和解码器分别优化。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### 视觉语言模型（VLM）压缩部署

In [ ]:
class SimpleViT(nn.Module):
    """简化视觉编码器"""
    def __init__(self, patch_size=16, in_channels=3, dim=128, n_layers=4, n_heads=4):
        super().__init__()
        self.patch_embed = nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=dim, nhead=n_heads, batch_first=True)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        h = self.patch_embed(x)
        h = h.flatten(2).transpose(1, 2)
        for layer in self.layers:
            h = layer(h)
        return self.norm(h)

class VisionLanguageModel(nn.Module):
    """简化VLM"""
    def __init__(self, dim=128, n_layers=4, n_heads=4, vocab_size=1000):
        super().__init__()
        self.vision_encoder = SimpleViT(dim=dim, n_layers=n_layers, n_heads=n_heads)
        self.projection = nn.Linear(dim, dim)
        self.llm = nn.Sequential(
            nn.Embedding(vocab_size, dim),
            nn.TransformerEncoder(
                nn.TransformerEncoderLayer(d_model=dim, nhead=n_heads, batch_first=True),
                num_layers=6
            ),
            nn.Linear(dim, vocab_size)
        )

    def forward(self, image, text_ids):
        vision_features = self.vision_encoder(image)
        projected = self.projection(vision_features)
        text_embed = self.llm[0](text_ids)
        combined = torch.cat([projected, text_embed], dim=1)
        llm_out = self.llm[1](combined)
        logits = self.llm[2](llm_out)
        return logits

vlm = VisionLanguageModel(dim=128, n_layers=4, n_heads=4, vocab_size=1000)

vit_params = sum(p.numel() for p in vlm.vision_encoder.parameters())
proj_params = sum(p.numel() for p in vlm.projection.parameters())
llm_params = sum(p.numel() for p in vlm.llm.parameters())
total_params = vit_params + proj_params + llm_params

print(f"=== VLM参数分布 ===")
print(f"视觉编码器: {vit_params:,} ({vit_params/total_params:.1%})")
print(f"投影层: {proj_params:,} ({proj_params/total_params:.1%})")
print(f"语言模型: {llm_params:,} ({llm_params/total_params:.1%})")
print(f"总计: {total_params:,}")

print(f"\n--- 各组件量化策略 ---")
print(f"视觉编码器: 可更激进量化(INT4)，因为视觉特征对精度不敏感")
print(f"投影层: 保持FP16，参数量小但影响大")
print(f"语言模型: INT4/INT8量化，与纯LLM策略相同")

vit_int4 = vit_params * 0.5 / (1024 * 1024)
proj_fp16 = proj_params * 2 / (1024 * 1024)
llm_int4 = llm_params * 0.5 / (1024 * 1024)
total_int4 = vit_int4 + proj_fp16 + llm_int4
total_fp32 = total_params * 4 / (1024 * 1024)

print(f"\n--- 混合量化后存储 ---")
print(f"FP32总存储: {total_fp32/1024/1024:.1f} MB")
print(f"混合量化存储: {total_int4/1024/1024:.1f} MB")
print(f"压缩比: {total_fp32/total_int4:.1f}x")

### 视觉Token压缩

In [ ]:
class TokenCompressor(nn.Module):
    """视觉Token压缩器: 减少视觉特征token数量"""
    def __init__(self, dim, target_tokens=16):
        super().__init__()
        self.target_tokens = target_tokens
        self.merge_weights = nn.Linear(dim, 1)

    def forward(self, vision_tokens):
        B, N, D = vision_tokens.shape
        if N <= self.target_tokens:
            return vision_tokens

        importance = self.merge_weights(vision_tokens).squeeze(-1)
        importance = F.softmax(importance, dim=-1)

        topk_indices = importance.topk(self.target_tokens, dim=-1).indices
        topk_indices = topk_indices.sort(dim=-1).values

        selected = torch.gather(vision_tokens, 1, topk_indices.unsqueeze(-1).expand(-1, -1, D))
        return selected

class AveragePoolingCompressor(nn.Module):
    """平均池化压缩器"""
    def __init__(self, target_tokens=16):
        super().__init__()
        self.target_tokens = target_tokens

    def forward(self, vision_tokens):
        B, N, D = vision_tokens.shape
        if N <= self.target_tokens:
            return vision_tokens
        group_size = N // self.target_tokens
        trimmed = vision_tokens[:, :group_size * self.target_tokens, :]
        pooled = trimmed.reshape(B, self.target_tokens, group_size, D).mean(dim=2)
        return pooled

dim = 128
n_original_tokens = 196
tokens = torch.randn(2, n_original_tokens, dim)

compressor_select = TokenCompressor(dim, target_tokens=16)
compressor_pool = AveragePoolingCompressor(target_tokens=16)

with torch.no_grad():
    selected = compressor_select(tokens)
    pooled = compressor_pool(tokens)

print(f"=== 视觉Token压缩 ===")
print(f"原始token数: {n_original_tokens}")
print(f"目标token数: 16")
print(f"压缩比: {n_original_tokens/16:.1f}x")
print(f"\n选择压缩后: {selected.shape}")
print(f"池化压缩后: {pooled.shape}")
print(f"\n对LLM推理的影响:")
original_attn_ops = n_original_tokens ** 2
compressed_attn_ops = 16 ** 2
print(f"  自注意力计算量: {original_attn_ops} -> {compressed_attn_ops} ({compressed_attn_ops/original_attn_ops*100:.1f}%)")
print(f"  KV Cache: {n_original_tokens} -> 16 tokens")
print(f"\n视觉Token压缩是VLM端侧部署的关键优化")